In [85]:
from pymarc import MARCReader
import pandas as pd
import requests
from dotenv import load_dotenv
import os
from pinecone import Pinecone
import time
import datetime
import json
import glob

In [86]:
# Load environment variables from .env file
load_dotenv()

# Get ISBN API key from environment variables
ISBN_API_KEY = os.getenv('ISBN_API_KEY')
pc = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))

In [87]:
today = datetime.datetime.now()

In [88]:
def extract_book_info(response_json):
    """
    Extract specific fields from ISBN API response and return as dictionary.
    
    Args:
        response_json (dict): The JSON response from the ISBN API
        
    Returns:
        dict: Dictionary containing extracted book information
    """
    extracted_info = {}
    
    # The response might have the book data nested under 'book' key
    book_data = response_json.get('book', response_json)
    
    # Extract publisher
    extracted_info['publisher'] = book_data.get('publisher', '')
    extracted_info['_id'] = book_data.get('isbn', '')
    
    # Extract synopsis (might be under 'synopsis', 'overview', or 'description')
    extracted_info['synopsis'] = (
        book_data.get('synopsis') or 
        book_data.get('overview') or 
        book_data.get('description') or 
        ''
    )
    
    # Extract title_long
    extracted_info['title'] = book_data.get('title_long', book_data.get('title', ''))
    
    # Extract pages (might be integer or string)
    pages = book_data.get('pages')
    if pages:
        try:
            extracted_info['pages'] = int(pages)
        except (ValueError, TypeError):
            extracted_info['pages'] = pages
    else:
        extracted_info['pages'] = ''
    
    # Extract date_published
    extracted_info['date_published'] = book_data.get('date_published', '')
    
    # Extract dewey_decimal
    #extracted_info['dewey_decimal'] = book_data.get('dewey_decimal', None)
    
    # Extract subjects (might be a list or string)
    subjects = book_data.get('subjects')
    if isinstance(subjects, list):
        extracted_info['subjects'] = subjects
    elif isinstance(subjects, str):
        extracted_info['subjects'] = [subjects]
    else:
        extracted_info['subjects'] = ''
    
    # Extract authors (might be a list or string)
    authors = book_data.get('authors')
    if isinstance(authors, list):
        extracted_info['authors'] = authors
    elif isinstance(authors, str):
        extracted_info['authors'] = [authors]
    else:
        extracted_info['authors'] = ''
    
    return extracted_info


In [89]:
def get_book_info(isbn):
    # https://isbndb.com/user/62794
    if isbn is None:
        return None

    # Clean the ISBN (remove any extra characters, spaces, etc.)
    clean_isbn = isbn.replace('-', '').replace(' ', '').strip()
    
    # API request to ISBN database
    url = f"https://api2.isbndb.com/book/{clean_isbn}"
    headers = {
        'User-Agent': 'python-requests/2.28.1',
        'Authorization': ISBN_API_KEY,  # Replace with your actual API key
        'Accept': '*/*'
    }
    
    
    try:
        start_time = time.time()
        response = requests.get(url, headers=headers)
        
        if response.status_code == 200:
            book_info = extract_book_info(response.json())
            end_time = time.time()
            if end_time - start_time > 1:
                return book_info
            else:
                time.sleep(1)
                return book_info
        else:
            print(f"Error Response: {response.text}")
            
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")


In [90]:
def check_none_values(record_dict):
    """
    Check each item in a single record dictionary for None values.
    Replaces None values with empty strings and prints what was changed.
    
    Args:
        record_dict (dict): A single record dictionary to check and modify
        
    Returns:
        dict: The modified dictionary with None values replaced by empty strings
    """
    none_count = 0
    
    try:
        for key, value in record_dict.items():
            if value is None or value == '':
                record_dict[key] = '-999'  # Replace None with empty string
            none_count += 1
    except Exception as e:
        print(f"Error checking record: {e}")
        return None

    return record_dict

In [91]:
index_name = "quickstart-py"
if not pc.has_index(index_name):
    pc.create_index_for_model(
        name=index_name,
        cloud="aws",
        region="us-east-1",
        embed={
            "model":"llama-text-embed-v2",
            "field_map": {
                "text": "title",
                "text": "authors",
                "text": "subjects",
                "text": "synopsis",
            }
        }
    )

In [92]:
# Path to your MARC file
marc_file = 'ExportBibJob605495USMarc.001'

In [93]:
existing_records = {}

# Get all JSON files in the data folder
json_files = glob.glob('data/book_records_*.json')

# Load and combine all records, using dict to automatically handle duplicates
for file in json_files:
    try:
        file_records = json.load(open(file))
        # Convert list of records to dict keyed by _id
        for r in file_records:
            if r is None:
                continue
            else:
                existing_records[r['_id']] = r
    except Exception as e:
        print(f"Error loading {file}: {e}")

# Convert back to list for consistency with rest of code
existing_records = list(existing_records.values())
print(f"Loaded {len(existing_records)} unique records")

Loaded 6373 unique records


In [94]:
isbns = []
with open(marc_file, 'rb') as file:
    reader = MARCReader(file, to_unicode=True, force_utf8=True)
    for record in reader:
        # Extract fields; use get_fields to handle multiple occurrences
        try:
            isbn = record['020']['a'] if record['020'] else ''
            if isbn:
                isbns.append(isbn)
        except Exception as e:
            print(f"Error processing record: {e}")
            continue

print(f"Total ISBNs found: {len(isbns)}")

Error processing record: 
Error processing record: 
Error processing record: 
Error processing record: 
Error processing record: 
Error processing record: 
Error processing record: 
Error processing record: 
Total ISBNs found: 11273


In [96]:
# Filter out ISBNs that already exist in records
new_isbns = []
existing_isbn_set = {r['_id'] for r in existing_records if r is not None}
for isbn in isbns:
    if isbn not in existing_isbn_set:
        new_isbns.append(isbn)

print(f"New ISBNs to process: {len(new_isbns)}")


New ISBNs to process: 8900


In [ ]:
# List to store records
records = []
num_errors = 0


for isbn in isbns:
    try:
        book_info = get_book_info(isbn)
        if book_info is None:
            continue
        else:
            book_info_cleaned = check_none_values(book_info)
            # Add the record to the list
            records.append(book_info_cleaned)
        
    except Exception as e:
        print(f"Error processing record {book_info.get('isbn', 'unknown')}: {e}")
        num_errors += 1

    if len(records) >= 4000:
        break

print(f"Number of records processed: {len(records)}")
print(f"Number of errors: {num_errors}")

KeyboardInterrupt: 

In [78]:
# Create filename with date
filename = 'data/book_records_{}.json'.format(today.strftime('%Y-%m-%d'))

# Save records to JSON file
with open(filename, 'w') as f:
    json.dump(records, f, indent=2)

print(f"Saved {len(records)} records to {filename}")

Saved 82 records to data/book_records_2025-08-10.json


In [79]:
clean_records = []
for r in records:
    new_r = check_none_values(r)
    if new_r is not None:
        clean_records.append(new_r)

print(len(clean_records))

82


In [80]:
# Target the index
dense_index = pc.Index(index_name)

In [81]:
problem_records = []
count = 0
for i in range(0, len(clean_records), 95):
    try:
        if count < 10:
            dense_index.upsert_records("example-namespace", clean_records[i:i+95])
            count += 1
        else:
            count = 0
            print(f"Sleeping for 60 seconds")
            time.sleep(60)
    except Exception as e:
        print(f"Error upserting records: {e}")
        print(f"Count: {count}")
        problem_records.append(clean_records[i:i+95])

In [82]:
problem_records_again = []
count = 0
for i in problem_records:
    try:
        if count < 5:
            dense_index.upsert_records("example-namespace", i)
            count += 1
        else:
            count = 0
            print(f"Sleeping for 60 seconds")
            time.sleep(60)
    except Exception as e:
        print(f"Error upserting records: {e}")
        print(f"Count: {count}")
        problem_records_again.append(i)

if len(problem_records_again) > 0:
    print(f"There are {len(problem_records_again)} records that need to be upserted again")
else:
    print("All records have been upserted successfully")

All records have been upserted successfully


In [83]:
time.sleep(10)

# View stats for the index
stats = dense_index.describe_index_stats()
print(stats)

{'dimension': 1024,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'example-namespace': {'vector_count': 6373}},
 'total_vector_count': 6373,
 'vector_type': 'dense'}


In [ ]:
query = "what is a book that has to do with the loch ness monster?"

In [ ]:
# Search the dense index and rerank results
reranked_results = dense_index.search(
    namespace="example-namespace",
    query={
        "top_k": 3,
        "inputs": {
            'text': query
        }
    },
    rerank={
        "model": "bge-reranker-v2-m3",
        "top_n": 3,
        "rank_fields": ["synopsis"]
    }   
)

# Print the reranked results
for hit in reranked_results['result']['hits']:
    print(hit)

{'_id': '144884763X',
 '_score': 0.9453993439674377,
 'fields': {'authors': ['Nikki Case', 'Martin Delrio'],
            'date_published': '2011-08-30',
            'pages': 64.0,
            'publisher': 'Rosen Central',
            'subjects': ['LOCH NESS MONSTER', 'MONSTERS_JUVENILE LITERATURE'],
            'synopsis': 'Since 1933, the Loch Ness monster has captured the '
                        'imaginations of countless children and adults alike. '
                        'This book offers further explanation of the mythical '
                        'creature, including its earliest sightings, history, '
                        'and its intriguing photographs. With the latest '
                        'research, this book stands as the most up-to-date '
                        'account of this elusive creature.',
            'title': 'Searching for the Loch Ness Monster (Mystery Explorers)'}}
{'_id': '0822516268',
 '_score': 0.07504072040319443,
 'fields': {'authors': ['Judith H